# 강의 04 · 실습 8 — 이미지 생성 파이프라인 · (4) 고난도 1 — 규격 자동 검사

## 1. 문제상황

- 디자이너가 뽑은 이미지 중에는 파일이 깨졌거나 크기가 지나치게 작은 것이 섞여 나옵니다.
- 그런 이미지는 사람이 볼 필요조차 없는데, 지금은 사람이 후보를 열어 본 뒤에야 걸러집니다.
- 사람은 하루에 수십 개의 후보를 보므로, 볼 필요 없는 후보가 섞일수록 판정 시간이 늘어납니다.
- 팀은 사람에게 보여 주기 전에 프로그램이 파일 형식과 크기를 먼저 검사하고, 불합격이면 한 번 더 뽑아 보기로 정했습니다.
- 그래도 불합격이면 경고를 붙여 사람에게 넘깁니다. 검사 없이 무한히 다시 뽑는 일은 없어야 합니다.

## 2. 문제와 목표

- **문제**: 규격에 못 미치는 후보가 사람의 판정 단계까지 올라옵니다. 규격 검사는 기계가 할 수 있는 일인데 사람이 하고 있습니다.
- **목표**: 생성 뒤에 프로그램이 파일 형식과 크기를 검사해 합격이면 사람에게 넘기고, 불합격이면 정해진 횟수 안에서 다시 뽑고, 횟수를 다 쓰면 경고를 붙여 넘기는 처리 흐름을 만듭니다. 템플릿 개정 되돌림(revise → design)은 그대로 둡니다.
    - 규격 검사: 파일 앞머리가 JPEG 또는 PNG 표시인지와 크기가 10,000바이트 이상인지를 보는 check 노드이며, 모델을 부르지 않습니다.
    - 정해진 횟수: 생성 시도 상한 2회이며 상태의 시도 횟수 키로 셉니다.
    - 상태 키 일곱 개: 주제, 템플릿, 지시문, 이미지 경로 목록, 검사 결과, 시도 횟수, 판정입니다. 사람의 답(「확정」 한 번)은 코드에 대본으로 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 주제 하나를 넣었을 때 규격 검사가 합격일 때만 사람의 판정을 기다리며 멈춥니다.
    - 사람에게 간 내용에 검사 결과와 시도 횟수가 들어 있습니다.
    - 「확정」이라고 답하면 END에 도달하는 것을 실행 결과에서 확인합니다. 불합격 분기는 크기 기준값을 파일 크기보다 크게 올려 실행하면 확인할 수 있습니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex08_s4_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 주제·템플릿·지시문·이미지 경로 목록·판정 키 다섯 개에 검사 결과(`checked`)와 생성 시도 횟수(`attempts`) 키 두 개를 더해 키 일곱 개를 선언합니다.
    - `images` 키에는 `add` 리듀서를 붙입니다.
2. **지시문 설계 노드를 만듭니다.**
    - design 노드는 상태의 `template` 뒤에 주제를 붙여 모델을 호출하고 `prompt` 키에 씁니다.
3. **이미지 생성 노드를 만듭니다.**
    - generate 노드는 `paint`로 이미지를 저장해 경로를 `images` 키에 이어 붙이고, `attempts` 키를 하나 늘립니다.
4. **검사 노드를 만듭니다.**
    - check 노드는 마지막 후보 파일을 읽어 앞머리가 JPEG(`FF D8`) 또는 PNG(`89 50 4E 47`) 표시인지와 크기가 `MIN_BYTES`(10,000바이트) 이상인지를 검사하고, 결과를 `checked` 키에 「합격」 또는 「불합격」으로 씁니다.
    - 판정 이유를 화면에 출력합니다.
5. **평가 노드를 만듭니다.**
    - review 노드는 `interrupt()`로 멈추고, 질문·후보 목록·검사 결과·시도 횟수를 사람에게 보냅니다.
    - 답을 `verdict` 키에 씁니다.
6. **템플릿 개정 노드를 만듭니다.**
    - revise 노드는 `template` 키를 `SPEC_V2`로 바꿉니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
7. **그래프에 노드를 등록합니다.**
    - 다섯 노드를 등록합니다.
8. **엣지를 연결합니다.**
    - START → design → generate → check를 고정 엣지로 연결합니다.
    - check 뒤에는 조건부 엣지 route_check를 놓습니다.
    - 합격이면 review, 불합격이고 `attempts`가 `MAX_ATTEMPTS`(2회)보다 작으면 generate, 그 밖에는 review입니다.
    - review 뒤에는 판정이 「재설계」이면 revise, 그 밖에는 END인 조건부 엣지를 추가하고 revise → design을 연결합니다.
9. **그래프를 컴파일하고 실행합니다.**
    - 체크포인터를 달아 컴파일하고, 주제 「비 내리는 밤 서울 골목의 LP 바 창가」와 템플릿 v1, `attempts` 0을 넣어 실행합니다.
    - `[check]` 줄과 멈춘 지점, 사람에게 간 내용을 출력한 뒤 `Command(resume="확정")`으로 종료합니다.
    - check 노드는 「[check] 형식=… 크기=… -> 합격/불합격」 줄을, 실행 셀은 「next = …」·「verdict = …」 줄과 생성 호출 횟수를 출력하며, 시도 상한에 닿으면 안내 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다. 이미지 모델을 부르는 노드와 `interrupt()`는 새 단계가 아니라 ② 노드 함수와 ⑤ 실행 단계 안에 들어갑니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class ImageState(TypedDict)`, `Annotated[list, add]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def check(state) -> dict`, `Path.read_bytes()`, `interrupt()` | 2, 3, 4, 5, 6 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(ImageState)`, `add_node` | 7 |
| ④ 엣지 연결 | 노드 사이의 순서와 두 분기를 정합니다 | `add_edge`, `add_conditional_edges` (두 곳) | 8 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, 멈춘 지점에서 사람의 답으로 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)` | 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 텍스트 모델과 이미지 모델을 준비합니다. 이미지 모델을 부르는 함수 `paint`도 여기서 정의합니다.

- API 키와 자격증명은 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다.
- `.env` 파일에는 다음 네 줄이 있어야 합니다. 값은 각자 발급받은 것을 넣습니다.

```
OPENAI_API_KEY=발급받은_키
GOOGLE_APPLICATION_CREDENTIALS=서비스_계정_키_파일의_경로
VERTEX_PROJECT=프로젝트_이름
VERTEX_LOCATION=리전_이름
```

- `paint` 호출 1회가 이미지 1장이고, 호출마다 비용이 듭니다. 생성한 이미지는 노트북 옆의 `out_images` 폴더에 저장됩니다.
- `show`는 후보 이미지 경로 목록을 화면에 표시하는 보조 함수입니다.
- 생성 호출 횟수는 전역 카운터 `paint_calls`로 셉니다.

In [ ]:
import os
import time
from operator import add
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from IPython.display import Image, display
from google import genai
from google.genai import types
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
for key in ("OPENAI_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "VERTEX_PROJECT", "VERTEX_LOCATION"):
    if not os.environ.get(key):
        raise SystemExit(f"agentic-ai 폴더의 .env 파일에 {key} 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
client = genai.Client(vertexai=True, project=os.environ["VERTEX_PROJECT"], location=os.environ["VERTEX_LOCATION"])
IMG_MODEL = "gemini-3.1-flash-lite-image"
OUT_DIR = Path("out_images")
OUT_DIR.mkdir(exist_ok=True)
paint_calls = 0


def paint(prompt: str, tag: str) -> str:
    """지시문을 이미지 모델에 보내 이미지 파일을 저장하고 경로를 돌려준다. 호출 1회 = 이미지 1장 = 비용 발생."""
    global paint_calls
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"]))
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [paint] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    paint_calls += 1
    part = [p for p in res.candidates[0].content.parts if p.inline_data][0].inline_data
    ext = "png" if "png" in part.mime_type else "jpg"
    path = OUT_DIR / f"{tag}_{paint_calls:02d}.{ext}"
    path.write_bytes(part.data)
    print(f"    [paint] {path.as_posix()} ({len(part.data)} bytes)")
    return path.as_posix()


def show(paths: list) -> None:
    """후보 이미지 경로 목록을 화면에 차례로 표시한다."""
    for i, p in enumerate(paths, 1):
        print(f"    후보 {i}: {p}")
        display(Image(filename=p, width=320))


print("모델 준비를 마쳤습니다. 이미지 저장 폴더:", OUT_DIR)

# 주어진 자료 — 지시문 템플릿 두 벌 (주제 문장을 뒤에 이어 붙여 언어 모델에 넣는다)
SPEC_V1 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 주제: ")
SPEC_V2 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 조명은 광원의 방향과 색온도를 한 구절로 적는다. 주제: ")


### 단계 ① — 상태 정의 (요구사항 1)

In [ ]:
# 여기에 단계 ①(상태 정의: 키 일곱 개)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5, 6)

- check 노드는 모델을 부르지 않습니다. 파일을 읽어 규칙으로 판정합니다.
- `MIN_BYTES`와 `MAX_ATTEMPTS`가 검사와 되돌림 상한의 원칙입니다. 노드는 판단하지 않고 원칙대로 움직입니다.

In [ ]:
# 여기에 단계 ②(노드 함수 다섯 개 정의)을 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 7)

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 8)

조건부 엣지가 두 곳입니다. check 뒤의 route_check와 review 뒤의 route입니다.

In [ ]:
# 여기에 단계 ④(엣지 연결과 판단 함수 두 개)을 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 9)

실행 설정은 `config = {"configurable": {"thread_id": "design-check"}}`이고, 멈춘 지점은 `graph.get_state(config).next`, 사람에게 간 내용은 `invoke` 반환값의 `out["__interrupt__"][0].value`에서 읽습니다.


In [ ]:
# 여기에 단계 ⑤(컴파일과 실행: 검사 결과 확인, 확정으로 종료)을 작성합니다.

## 7. 실행 결과 확인

1. `[check]` 줄에 형식과 크기가 출력되고 결과가 합격입니다. 그 뒤에야 `next = ('review',)`가 출력됩니다.
2. 사람에게 간 내용에 검사 결과 「합격」과 시도 1이 들어 있습니다. 사람은 규격에 맞는 후보만 봅니다.
3. 재개 뒤 `verdict = 확정`, `next = ()`, 생성 호출 1회입니다.
4. 불합격 분기를 보려면 `MIN_BYTES`를 파일 크기보다 크게 올리고 다시 실행합니다. `[check]` 불합격 → generate 재실행 → 두 번째 `[check]` → 시도 상한 안내 → review 순서로 출력됩니다. 이 확인은 생성 호출이 2회 더 들므로 필요할 때만 합니다.